# Infra-Bench — CROMA_base Full Fine-Tune (spatial split, 3 seeds with resume)

Full fine-tuning evaluation of CROMA_base on Infra-Bench, complementing
the linear-probe result reported in `croma/lp_1.0x.ipynb`.
Motivated by coauthor feedback (Konrad Wessels) and GEO-Bench conventions:
full fine-tuning is the headline evaluation for EO FMs, not just linear
probing.

## What's different vs. the LP notebook

| Aspect | Linear probe v2 | Fine-tune v1 |
|---|---|---|
| Backbone | Frozen (`self.model.eval()` + `requires_grad = False`) | Fully trainable (`freeze=False` skips both) |
| Optimizer param scope | Head only (~845 params) | Head + full joint encoder (~86M) |
| Learning rate | 1e-3 (head only) | **1e-4** (all params) per Fuller et al. 2023 |
| Weight decay | 1e-4 | **0.05** per Fuller et al. 2023 |
| LR schedule | None | **Cosine annealing, no warmup** per Fuller et al. 2023 |
| Epochs | 25 | 25 (unchanged for comparability) |
| Batch size | 16 | 16 (unchanged) |
| Class weight cap | 10.0 | 10.0 (unchanged) |
| Seeds | 3 (314, 271, 161) | 3 (314, 271, 161) |
| Spatial split artifact | `asset_id_to_split_v1.parquet` | same (direct LP comparability) |
| BEST_CKPT_BEFORE_TEST | present | preserved |
| Per-epoch `metrics.jsonl` append | present | preserved (adds `current_lr`) |
| Per-epoch `checkpoint_final.pt` write | end only | **every epoch** (resume support) |
| Resume-from-checkpoint | N/A | **yes** — detects existing `checkpoint_final.pt` |

## Runtime hardware: Colab H100

CROMA fine-tune runs on **Colab H100** (not the L4 used for LP + Satlas
fine-tune). The 24h+ per-seed estimate from earlier LP-based extrapolation
was L4-throughput; H100's higher throughput on the CROMA joint encoder
should reduce per-seed wall clock substantially. The smoke cell's
extrapolation reports the actual figure — set Colab runtime to H100
before running any cells.

## Compute constraints and mitigations

Both mitigations from the original L4 plan are preserved — they remain
useful regardless of GPU tier:

1. **3 seeds** (SEEDS=[314, 271, 161]) matching the S1/S2/Prithvi
   protocol. Resume-from-checkpoint below is what makes 3 seeds on
   H100 tractable within a Colab Pro+ compute budget.
2. **Resume-from-checkpoint**. `train_one_seed` detects an existing
   `checkpoint_final.pt` in the seed's ckpt_dir and resumes from the
   next epoch. Every epoch, `checkpoint_final.pt` is overwritten with
   the current model + optimizer + scheduler + history state. If a
   Colab session dies mid-run, re-running the multi-seed cell picks
   up where it left off. `metrics.jsonl` is preserved across resumes
   (not truncated on restart).

## What's unchanged

- Dataset pipeline (`CROMADataset`, `MultiSectorLabelWrapper`, `SubsetView`)
- Spatial split artifact and old-vs-new-split diagnostic
- Data normalization (percentile 2/98 → [0, 1], both S1 and S2 modalities)
- Band selection (12 S2 bands + 2 S1, joint 14-ch input)
- Per-sector F1 v2 definition (`_per_sector_v2`)
- Evaluation function and confusion-matrix machinery

## Additional aggregate JSON fields

- `finetune_protocol`: dict recording the exact hyperparameters used
- `peak_gpu_gb` per seed: `torch.cuda.max_memory_allocated()` during training
- `wall_time_s` per seed: total wall clock of `train_one_seed` (sum across
  resume sessions if the run resumed)
- `resumed_epochs` per seed: epoch count that was recovered from a prior
  session's `checkpoint_final.pt`, if any (else 0)

## Outputs (only after `SMOKE_ONLY=False`)

- `/.../results/fm_eval_croma_finetune_v1/croma_finetune_v1_seed314_results.json`
- `/.../results/fm_eval_croma_finetune_v1/croma_finetune_v1_aggregate.json`
- `/.../results/fm_eval_croma_finetune_v1/confusion_matrix_croma_finetune_v1_aggregate.png`
- `croma_finetune_v1_seed314_finetune/checkpoint_best.pt`, `checkpoint_final.pt`, `metrics.jsonl`

## Smoke test

Default `SMOKE_ONLY=True` runs 2 epochs on seed 314 using a **distinct
run_name** (`CROMA_SMOKE_seed314_finetune`), so smoke's checkpoints don't
collide with the full-run's resume detection. Reports:
- Trainable-parameter count (should be ~86M, not ~845)
- Peak GPU memory after one train step
- Per-epoch LR trajectory (scheduler built for full 25 epochs)
- Divergence check on train_loss
- Extrapolated total wall clock

Flip to `SMOKE_ONLY = False` in the smoke cell to launch the full run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [12]:
%%capture
!pip install -q einops huggingface_hub scikit-learn pyproj pyarrow

In [ ]:
import os, sys, urllib.request
from pathlib import Path

CROMA_DIR     = Path('/content/croma_repo')
CROMA_DIR.mkdir(exist_ok=True)
USE_CROMA_PY  = CROMA_DIR / 'use_croma.py'
USE_CROMA_URL = 'https://raw.githubusercontent.com/antofuller/CROMA/main/use_croma.py'

if not USE_CROMA_PY.exists():
    print(f'Fetching {USE_CROMA_URL} ...')
    try:
        urllib.request.urlretrieve(USE_CROMA_URL, USE_CROMA_PY)
        print(f'  -> {USE_CROMA_PY}  ({USE_CROMA_PY.stat().st_size:,} bytes)')
    except Exception as e:
        print(f'  direct download failed ({e}); falling back to git clone')
        import subprocess
        subprocess.check_call(['git', 'clone', '--depth', '1', '-q',
                               'https://github.com/antofuller/CROMA.git',
                               str(CROMA_DIR)])
else:
    print(f'Already have {USE_CROMA_PY}')

from huggingface_hub import hf_hub_download
WEIGHT_CANDIDATES = ['CROMA_base.pt', 'croma_base.pt', 'CROMA_base.pth']
CROMA_WEIGHTS = None
last_err = None
for candidate in WEIGHT_CANDIDATES:
    try:
        CROMA_WEIGHTS = hf_hub_download(repo_id='antofuller/CROMA', filename=candidate)
        print(f'Downloaded weights: {candidate} -> {CROMA_WEIGHTS}')
        break
    except Exception as e:
        last_err = e
        continue
if CROMA_WEIGHTS is None:
    raise RuntimeError(f'No CROMA_base weights on HF. Last error: {last_err}')

if str(CROMA_DIR) not in sys.path:
    sys.path.insert(0, str(CROMA_DIR))
from use_croma import PretrainedCROMA
print('use_croma.PretrainedCROMA imported OK')


In [ ]:
# extract curation code zip to get curation.utils.spatial_blocking.
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# import the spatial split loader. if the zip is older than Phase 1 (no
# spatial_blocking.py yet), fall back to a slim local definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import (zip is pre-Phase-1): {e}. Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


In [ ]:
import os
from pathlib import Path

DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL      = '/content/datasets'
OUTPUT_DIR          = f'{DRIVE_ROOT}/results/fm_eval_croma_finetune_v1'
# the split artifact ships inside the code zip, so it resolves from the
# extracted repo. Drive stays a fallback for setups that still stage it there.
_drive_split = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
try:
    from curation.paths import SPLIT_ARTIFACT as _repo_split
    SPLIT_ARTIFACT_PATH = str(_repo_split) if _repo_split.exists() else _drive_split
except Exception:
    SPLIT_ARTIFACT_PATH = _drive_split
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':                  'water.water_works',   # legacy manifest tag
    'water.water_works':                      'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

# CROMA input: 12 S2 bands + 2 S1 (VV, VH). joint 14-channel encoder.
# CROMADataset handles the assembly.
PERC_LO, PERC_HI = 2.0, 98.0

# CROMA S2 channel remap (unchanged from LP). our .npy stores 7 S2 bands
# at indices 0..6; CROMA expects a 12-slot layout; -1 = zero-fill.
CROMA_S2_FROM_OURS = [-1, 2, 1, 0, -1, -1, -1, 3, 4, 5, 6, -1]
S1_INDICES_IN_OURS = [7, 8]


# ---- fine-tune hyperparameters (Fuller et al. NeurIPS 2023) ----
# cosine annealing over full training, NO warmup. weight decay 0.05
# (5x higher than SatlasPretrain — Fuller's paper cites this as
# CROMA-specific tuning for joint-encoder stability).
IMAGE_SIZE       = 224
FT_EPOCHS        = 25
FT_BATCH         = 16
FT_LR            = 1e-4
FT_WD            = 0.05           # <-- CROMA-specific: 5× S1/S2
FT_WARMUP_STEPS  = 0              # <-- CROMA-specific: no warmup
WEIGHT_CAP       = 10.0
SEEDS            = [314, 271, 161]
# the aggregate write below is gated on matching this exactly
FULL_PROTOCOL_SEEDS            = [314, 271, 161]
RUN_NAME_PREFIX  = 'croma_finetune_v1'
CONFUSION_CMAP   = 'Oranges'
CONFUSION_TITLE_PREFIX = 'CROMA fine-tune v1'

# resume-from-checkpoint: if the ckpt_dir for a seed contains
# checkpoint_final.pt from a prior (interrupted) session, pick up where
# we left off. see train_one_seed for the load logic.
RESUME_FROM_CHECKPOINT = True

# auto-skip smoke check on resume. when any full-run seed dir already contains
# a checkpoint_final.pt, we're in a post-disconnect reconnect scenario — the
# smoke check would just waste ~10 min on a distinct SMOKE_ dir before the
# multi-seed cell can even start. set False to force smoke regardless.
AUTO_SKIP_SMOKE_IF_RESUMING = True

FINETUNE_PROTOCOL = {
    'backbone_lr':      FT_LR,
    'head_lr':          FT_LR,
    'weight_decay':     FT_WD,
    'scheduler':        'cosine_no_warmup',
    'warmup_steps':     FT_WARMUP_STEPS,
    'llrd':             None,
    'autocast':         False,
    'grad_checkpointing': False,
    'grad_accum_steps': 1,
    'epochs':           FT_EPOCHS,
    'batch_size':       FT_BATCH,
    'effective_batch_size': FT_BATCH,
    'class_weight_cap': WEIGHT_CAP,
    'seeds':            list(SEEDS),
    'unfreeze_scope':   'all_backbone_params_joint_encoder',
    'resume_supported': True,
    'runtime_gpu':      'A100 (Colab)',
    'reference':        'Fuller et al. NeurIPS 2023 (CROMA)',
}


def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f'Output dir:            {OUTPUT_DIR}')
print(f'Split artifact:        {SPLIT_ARTIFACT_PATH}')
print(f'Percentile norm:       [{PERC_LO}, {PERC_HI}] -> [0, 1] (S1 + S2)')
print(f'Training seeds:        {SEEDS}   (3-seed protocol; resume-from-checkpoint enabled)')
print(f'Fine-tune protocol:    {FT_EPOCHS} epochs, batch {FT_BATCH}, lr {FT_LR}, '
      f'wd {FT_WD}, cosine (no warmup)')
print(f'Class weight cap:      {WEIGHT_CAP}')
print(f'Resume from ckpt:      {RESUME_FROM_CHECKPOINT}')


In [ ]:
# same materialize / discovery as v1.
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')

def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m: continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS: continue
        key = (region, sector)
        if key in seen: continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir
    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            return None
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [COPY]   {region:<22s} {sector:<10s} ({n} tiles)')
        return target_dir
    t0 = time.time()
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
ready = []
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local and (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))
print(f'\nReady: {len(ready)} (region, sector) pairs')


In [ ]:
# same CROMADataset + wrappers as v1 — split assignment happens later.
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


def percentile_normalize(arr, lo=PERC_LO, hi=PERC_HI):
    img = arr.astype(np.float32)
    low = np.percentile(img, lo); high = np.percentile(img, hi)
    if high <= low:
        return np.clip(img / 255.0, 0.0, 1.0)
    return np.clip((img - low) / (high - low), 0.0, 1.0)


class CROMADataset(Dataset):
    def __init__(self, dataset_root,
                 croma_s2_from_ours=CROMA_S2_FROM_OURS,
                 s1_indices=S1_INDICES_IN_OURS,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.croma_s2_from_ours = list(croma_s2_from_ours)
        self.s1_indices = list(s1_indices)
        self.our_s2_indices_used = sorted(i for i in self.croma_s2_from_ours if i >= 0)
        self.max_required_band = max(self.our_s2_indices_used + self.s1_indices)
        self.allowed = set(allowed_asset_types)
        with (self.dataset_root / 'manifest.json').open() as f:
            manifest = json.load(f)
        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'
        records = []
        for r in records_in:
            at = r.get('asset_type')
            if not at or at not in self.allowed: continue
            img_file = r.get('image_file')
            if not img_file: continue
            p = images_dir / img_file
            if not p.exists(): continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < self.max_required_band + 1: continue
            except Exception:
                continue
            records.append(_Record(path=p, asset_id=str(r.get('asset_id', p.stem)),
                                   asset_type=at))
        if not records:
            raise RuntimeError(f'no records: {dataset_root}')
        self.records = records

    def __len__(self):
        return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)
        H, W = arr.shape[1], arr.shape[2]
        s2_ours = arr[self.our_s2_indices_used, :, :]
        s2_ours = percentile_normalize(s2_ours)
        ours_pos = {our_idx: pos for pos, our_idx in enumerate(self.our_s2_indices_used)}
        s2_croma = np.zeros((12, H, W), dtype=np.float32)
        for slot, our_idx in enumerate(self.croma_s2_from_ours):
            if our_idx >= 0:
                s2_croma[slot] = s2_ours[ours_pos[our_idx]]
        s1 = arr[self.s1_indices, :, :].astype(np.float32)
        s1 = percentile_normalize(s1)
        return np.concatenate([s2_croma, s1], axis=0)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)
        return {'image': torch.from_numpy(img),
                'asset_id': r.asset_id, 'asset_type': r.asset_type}


class MultiSectorLabelWrapper(Dataset):
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base; self.region = region; self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels, self.asset_ids = [], [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None: continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])
            self.asset_ids.append(r.asset_id)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']
        img = F.interpolate(img.unsqueeze(0),
                            size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False).squeeze(0)
        return {'image': img, 'label': self.labels[idx],
                'asset_id': sample['asset_id'],
                'region': self.region, 'sector': self.sector}


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base; self.indices = indices
        self.region = base.region; self.sector = base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


# build per-cell datasets.
source_datasets = {}
for region, sector, local in ready:
    base = CROMADataset(local)
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds): source_datasets[(region, sector)] = ds
print(f'Built {len(source_datasets)} cell datasets')


In [ ]:
# load the spatial split artifact and slice each cell into train/val/test
# SubsetViews accordingly.
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  splits distribution: {Counter(asset_to_split.values())}')

splits = {}
n_unmapped = 0
for key, ds in source_datasets.items():
    region, sector = key
    tr_idx, va_idx, te_idx = [], [], []
    for i in range(len(ds)):
        asset_id = ds.asset_ids[i]
        sp = asset_to_split.get(asset_id)
        if sp == 'train':   tr_idx.append(i)
        elif sp == 'val':   va_idx.append(i)
        elif sp == 'test':  te_idx.append(i)
        else:                n_unmapped += 1
    splits[key] = {
        'train': SubsetView(ds, tr_idx),
        'val':   SubsetView(ds, va_idx),
        'test':  SubsetView(ds, te_idx),
    }

if n_unmapped > 0:
    print(f'NOTE: {n_unmapped} tiles in source_datasets have no split assignment '
          '(likely AlphaEarth-missing; will be excluded from train/val/test).')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])
print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')
print(f'  total in splits: {len(train_global) + len(val_global) + len(test_global)}')


In [ ]:
# diagnostic: regenerate the old v1 random-stratified split inside the
# notebook for an exact-match comparison. same logic as v1 CROMA's
# stratified_split (per-cell, by-class, seed=42).
print('=' * 76)
print('Diagnostic: train/val/test transition table (old random -> new spatial)')
print('=' * 76)

def old_stratified_split(dataset_labels, train_frac=0.7, val_frac=0.15, seed=42):
    by_class = {}
    for i, label in enumerate(dataset_labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


# walk cells in the same sorted (region, sector) order as v1 builds source_datasets.
old_split = {}
for key in sorted(source_datasets):
    ds = source_datasets[key]
    tr, va, te = old_stratified_split(ds.labels)
    for i in tr: old_split[ds.asset_ids[i]] = 'train'
    for i in va: old_split[ds.asset_ids[i]] = 'val'
    for i in te: old_split[ds.asset_ids[i]] = 'test'

# compare against asset_to_split (the new spatial split).
common_ids = set(old_split) & set(asset_to_split)
print(f'Comparing on {len(common_ids):,} tiles in both old and new splits')
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], asset_to_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')


In [ ]:
import torch.nn as nn

class CROMABackbone(nn.Module):
    NAME = 'croma_base'
    def __init__(self, weights_path, image_resolution=IMAGE_SIZE, freeze=True):
        super().__init__()
        self.model = PretrainedCROMA(
            pretrained_path=weights_path, size='base',
            modality='both', image_resolution=image_resolution,
        )
        if freeze:
            self.model.eval()
            for p in self.model.parameters():
                p.requires_grad = False
        self.feature_dim = self._infer_feature_dim()

    def _split(self, x):
        return x[:, :12], x[:, 12:14]

    def _infer_feature_dim(self):
        device = next(self.model.parameters()).device
        dummy = torch.zeros(1, 14, IMAGE_SIZE, IMAGE_SIZE, device=device)
        s2, s1 = self._split(dummy)
        with torch.no_grad():
            out = self.model(SAR_images=s1, optical_images=s2)
        if not isinstance(out, dict) or 'joint_GAP' not in out:
            raise RuntimeError(f'Unexpected CROMA output keys: {list(out.keys()) if isinstance(out, dict) else type(out)}')
        d = out['joint_GAP'].shape[-1]
        print(f'  feature_dim (joint_GAP) = {d}')
        return d

    def forward(self, x):
        s2, s1 = self._split(x)
        out = self.model(SAR_images=s1, optical_images=s2)
        return out['joint_GAP']


class InfraBenchClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(backbone.feature_dim, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


print('CROMABackbone + InfraBenchClassifier defined.')


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=WEIGHT_CAP):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """Per-sector F1 v2 — matches LP notebook exactly."""
    cm = np.array(cm)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition, matches LP).'
        )
    return result


def train_one_seed(seed, *, train_set, val_set, test_set,
                   num_epochs=None, scheduler_total_epochs=None,
                   run_name_override=None, allow_resume=None):
    """Full fine-tune training loop with resume-from-checkpoint support.

    Parameters
    ----------
    num_epochs : int or None
        Training-loop iteration count. Defaults to FT_EPOCHS.
    scheduler_total_epochs : int or None
        Scheduler length in epochs. Defaults to num_epochs. Smoke passes
        FT_EPOCHS so the LR trajectory reflects the real 25-epoch schedule.
    run_name_override : str or None
        Override the ckpt_dir name. Smoke passes a distinct value so its
        checkpoints don't collide with the full-run's resume detection.
    allow_resume : bool or None
        If True and checkpoint_final.pt exists in ckpt_dir, resume from it.
        Defaults to RESUME_FROM_CHECKPOINT.

    Resume behavior:
      - If checkpoint_final.pt exists, load model + optimizer + scheduler
        + history + best_val_f1 + best_epoch state.
      - Continue from epoch ckpt['epoch'] + 1 through num_epochs.
      - `metrics.jsonl` is NOT truncated on resume (preserves history).
      - If no checkpoint exists (fresh start), truncate metrics.jsonl and
        train from epoch 1.
    """
    if num_epochs is None:
        num_epochs = FT_EPOCHS
    if scheduler_total_epochs is None:
        scheduler_total_epochs = num_epochs
    if allow_resume is None:
        allow_resume = RESUME_FROM_CHECKPOINT
    set_seed(seed)
    print(f'\n--- seed {seed}  fine-tune ---')
    print(f'    training loop: {num_epochs} epochs   scheduler length: {scheduler_total_epochs} epochs')

    backbone = CROMABackbone(weights_path=CROMA_WEIGHTS, image_resolution=IMAGE_SIZE, freeze=False)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Total params:     {total_params:>13,}')
    print(f'  Trainable params: {trainable_params:>13,}')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=FT_BATCH, shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True,
                              generator=g)
    val_loader   = DataLoader(val_set,   batch_size=FT_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=FT_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # CROMA-specific: AdamW, wd=0.05, per Fuller et al.
    optimizer = AdamW(model.parameters(), lr=FT_LR, weight_decay=FT_WD)

    # CROMA-specific: pure cosine, no warmup. per-batch stepping.
    steps_per_epoch = len(train_loader)
    total_steps     = scheduler_total_epochs * steps_per_epoch
    scheduler       = CosineAnnealingLR(optimizer, T_max=max(total_steps, 1))

    # ==== Dual-write ckpts (local scratch + Drive mirror) ====
    # Colab Drive fuse buffers writes; if the runtime disconnects
    # before sync, writes are lost. write to /content/ (fast,
    # reliable fsync) and mirror to Drive after each save.
    import shutil as _shutil
    run_name = run_name_override or f'{RUN_NAME_PREFIX}_seed{seed}_finetune'
    
    # local scratch (primary I/O during training)
    ckpt_dir = Path('/content/ckpts_scratch') / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt     = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt    = ckpt_dir / 'checkpoint_final.pt'
    metrics_jsonl = ckpt_dir / 'metrics.jsonl'
    
    # Drive mirror (cross-session persistence)
    drive_ckpt_dir = Path(OUTPUT_DIR) / run_name
    drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
    drive_best_ckpt     = drive_ckpt_dir / 'checkpoint_best.pt'
    drive_final_ckpt    = drive_ckpt_dir / 'checkpoint_final.pt'
    drive_metrics_jsonl = drive_ckpt_dir / 'metrics.jsonl'
    
    # rehydrate local from Drive if VM was recycled (local disk cleared).
    # this is what makes cross-session resume work: Drive is the durable
    # store, local is the fast working copy.
    if not final_ckpt.exists() and drive_final_ckpt.exists():
        _shutil.copy(drive_final_ckpt, final_ckpt)
        if drive_best_ckpt.exists():
            _shutil.copy(drive_best_ckpt, best_ckpt)
        if drive_metrics_jsonl.exists():
            _shutil.copy(drive_metrics_jsonl, metrics_jsonl)
        print(f'  [DUAL-WRITE] rehydrated local from Drive: {drive_ckpt_dir.name}')
    # ---- resume-from-checkpoint logic ----
    history = []
    best_val_f1 = -1.0
    best_epoch = -1
    start_epoch = 0
    resumed_epochs = 0

    if allow_resume and final_ckpt.exists():
        ckpt = torch.load(final_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        history       = ckpt.get('history', [])
        # --- rewrite metrics.jsonl from history (source of truth). ---
        # discards any entries whose ckpt save didn't complete before a
        # disconnect (those epochs get re-trained anyway).
        metrics_jsonl.write_text('', encoding='utf-8')
        with metrics_jsonl.open('a', encoding='utf-8') as _mf:
            for _entry in history:
                _mf.write(json.dumps(_entry) + '\n')
            _mf.flush()
            os.fsync(_mf.fileno())
        best_val_f1   = ckpt.get('best_val_f1', -1.0)
        best_epoch    = ckpt.get('best_epoch', -1)
        start_epoch   = int(ckpt.get('epoch', 0))
        resumed_epochs = start_epoch
        print(f'  [RESUME] Loaded {final_ckpt.name}: '
              f'completed {start_epoch}/{num_epochs} epochs, '
              f'best_val_f1={best_val_f1:.4f} at epoch {best_epoch}')
        print(f'  [RESUME] metrics.jsonl NOT truncated; will append from epoch {start_epoch + 1}')
    else:
        # fresh start: truncate metrics.jsonl for a clean run
        metrics_jsonl.write_text('', encoding='utf-8')

    if start_epoch >= num_epochs:
        print(f'  [RESUME] Seed already completed ({start_epoch} >= {num_epochs}); '
              f'skipping training and going straight to test eval.')

    t_seed_start = time.time()

    for epoch in range(start_epoch, num_epochs):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()   # per-batch cosine step
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        current_lr = optimizer.param_groups[0]['lr']
        entry = {
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'current_lr': current_lr,
            'time_s': time.time() - t0,
        }
        history.append(entry)
        with metrics_jsonl.open('a', encoding='utf-8') as f:
            f.write(json.dumps(entry) + '\n'); f.flush(); os.fsync(f.fileno())
        _shutil.copy(metrics_jsonl, drive_metrics_jsonl)

        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch = epoch + 1
            marker = ' *'
            torch.save({'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'val_macro_f1': val['macro_f1'],
                        'history': history}, best_ckpt)
            _shutil.copy(best_ckpt, drive_best_ckpt)
        # CROMA-specific: write checkpoint_final.pt EVERY epoch (resume support).
        # bounds recovery to at most 1 epoch of lost work if session dies.
        torch.save({'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'history': history,
                    'best_val_f1': best_val_f1,
                    'best_epoch': best_epoch}, final_ckpt)

        _shutil.copy(final_ckpt, drive_final_ckpt)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  lr={current_lr:.2e}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    # ============== BEST_CKPT_BEFORE_TEST (preserved from LP) ==========
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'  [BEST_CKPT_BEFORE_TEST] restored epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_macro_f1"]:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)

    wall_time_s = time.time() - t_seed_start
    peak_gpu_gb = (torch.cuda.max_memory_allocated(DEVICE) / 1e9
                   if torch.cuda.is_available() else 0.0)

    return {
        'run_name':      run_name,
        'backbone':      backbone.NAME,
        'condition':     'full_finetune',
        'num_epochs':    num_epochs,
        'scheduler_total_epochs': scheduler_total_epochs,
        'seed':          seed,
        'best_val_f1':   best_val_f1,
        'best_epoch':    best_epoch,
        'tail_mean_f1':  float(np.mean(tail)),
        'tail_std_f1':   float(np.std(tail)),
        'history':       history,
        'test':          test,
        'tested_with':   tested_with,
        'peak_gpu_gb':   float(peak_gpu_gb),
        'wall_time_s':   float(wall_time_s),
        'resumed_epochs': int(resumed_epochs),
        'total_params':      int(total_params),
        'trainable_params':  int(trainable_params),
    }


print('Fine-tune training infrastructure ready.')
print('  - AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)')
print('  - Pure CosineAnnealingLR, no warmup, per-batch stepping')
print('  - scheduler_total_epochs decoupled from num_epochs')
print('  - Resume-from-checkpoint via checkpoint_final.pt (per-epoch write)')
print('  - Tracks peak_gpu_gb + wall_time_s + resumed_epochs per seed')


In [ ]:
# ============================================================================
# SMOKE CHECK — 2 epochs on seed 314 using a distinct run_name so the
# smoke's checkpoint doesn't collide with the full run's resume detection.
# see S1 notebook for the schedule-preview rationale.
# ============================================================================
SMOKE_ONLY   = False
SMOKE_EPOCHS = 2
SMOKE_SEED   = SEEDS[0]
SMOKE_RUN_NAME = f'CROMA_SMOKE_seed{SMOKE_SEED}_finetune'   # distinct from full-run ckpt dir

# ---- auto-skip on resume ---------------------------------------------------
# if any full-run seed dir already contains checkpoint_final.pt, this is a
# reconnect after a disconnect — running smoke would waste ~10 min on a
# distinct SMOKE_ ckpt dir before the multi-seed cell can pick up the
# interrupted run. bypass smoke and go straight to resume.
_full_run_ckpts_present = any(
    (Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{s}_finetune' / 'checkpoint_final.pt').exists()
    for s in SEEDS
)
if AUTO_SKIP_SMOKE_IF_RESUMING and _full_run_ckpts_present:
    print('=' * 76)
    print('AUTO_SKIP_SMOKE: existing full-run checkpoint_final.pt detected in one')
    print(f'or more of SEEDS={SEEDS} ckpt dirs. Interpreting this as a reconnect')
    print('after a disconnect — skipping the 2-epoch smoke check.')
    print()
    print('SMOKE_ONLY set to False; the multi-seed cell below will resume.')
    print('=' * 76)
    SMOKE_ONLY = False
    smoke_result = None
else:

    print(f'SMOKE_ONLY = {SMOKE_ONLY}')
    print(f'Smoke check: {SMOKE_EPOCHS} epochs on seed {SMOKE_SEED}, '
          f'lr={FT_LR}, wd={FT_WD}, cosine (no warmup)')
    print(f'Scheduler built for FULL {FT_EPOCHS}-epoch protocol (previews real LR trajectory).')
    print(f'Smoke ckpt dir: {SMOKE_RUN_NAME}  (isolated from full-run resume)')
    print('=' * 76)

    print('\n[1] Instantiate fine-tune model, verify forward pass on one batch...')
    set_seed(SMOKE_SEED)
    sb = CROMABackbone(weights_path=CROMA_WEIGHTS, image_resolution=IMAGE_SIZE, freeze=False)
    sm = InfraBenchClassifier(sb, num_classes=len(CLASS_NAMES)).to(DEVICE)
    _total  = sum(p.numel() for p in sm.parameters())
    _train  = sum(p.numel() for p in sm.parameters() if p.requires_grad)
    print(f'  Total params:     {_total:>13,}')
    print(f'  Trainable params: {_train:>13,}   (expect ~86M for full CROMA fine-tune)')
    assert _train > 80_000_000, (
        f'Trainable param count {_train:,} looks too small for full fine-tune; '
        f'expected ~86M. Backbone freeze may not have been disabled.'
    )

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)

    smoke_loader = DataLoader(train_global, batch_size=FT_BATCH, shuffle=True,
                              num_workers=0, collate_fn=collate)
    batch = next(iter(smoke_loader))
    img = batch['image'].to(DEVICE)
    lbl = batch['label'].to(DEVICE)
    weights = compute_class_weights(train_global).to(DEVICE)
    crit  = nn.CrossEntropyLoss(weight=weights)
    opt   = AdamW(sm.parameters(), lr=FT_LR, weight_decay=FT_WD)

    sm.train()
    opt.zero_grad()
    loss = crit(sm(img), lbl)
    loss.backward()
    opt.step()
    print(f'  One-step loss: {loss.item():.4f}  (finite: {torch.isfinite(loss).item()})')
    assert torch.isfinite(loss).item(), 'Loss is NaN/Inf on first step — abort.'

    if torch.cuda.is_available():
        peak_after_step = torch.cuda.max_memory_allocated(DEVICE) / 1e9
        print(f'  Peak GPU after 1 step: {peak_after_step:.2f} GB')
        if peak_after_step > 75.0:
            print(f'  !! WARNING: {peak_after_step:.2f} GB is close to A100 40 GB limit.')
        del sm, sb, opt

    print(f'\n[2] Running {SMOKE_EPOCHS}-epoch smoke on seed {SMOKE_SEED} '
          f'(schedule built for {FT_EPOCHS} epochs)...')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)

    # smoke uses isolated run_name and disables resume so smoke never picks
    # up state from a previous smoke run — always a fresh 2-epoch check.
    smoke_result = train_one_seed(
        SMOKE_SEED,
        train_set=train_global,
        val_set=val_global,
        test_set=test_global,
        num_epochs=SMOKE_EPOCHS,
        scheduler_total_epochs=FT_EPOCHS,
        run_name_override=SMOKE_RUN_NAME,
        allow_resume=False,
    )

    hist = smoke_result['history']
    train_losses = [h['train_loss'] for h in hist]
    val_f1s      = [h['val_macro_f1'] for h in hist]
    per_epoch_s  = [h['time_s'] for h in hist]
    per_epoch_lr = [h['current_lr'] for h in hist]

    print('\n' + '=' * 76)
    print('SMOKE SUMMARY')
    print('=' * 76)
    print(f'  train_loss trajectory:   {[f"{l:.4f}" for l in train_losses]}')
    print(f'  val_macro_f1 trajectory: {[f"{f:.4f}" for f in val_f1s]}')
    print(f'  end-of-epoch LR:         {[f"{lr:.2e}" for lr in per_epoch_lr]}')
    print(f'  time per epoch:          {[f"{t:.0f}s" for t in per_epoch_s]}')
    print(f'  peak GPU during train:   {smoke_result["peak_gpu_gb"]:.2f} GB')
    print(f'  wall time (smoke seed):  {smoke_result["wall_time_s"]:.0f} s')

    # CROMA has NO warmup, so ep1 LR is already at FT_LR × (1 - very small).
    # cosine's initial slope is 0 (derivative is -sin at t=0), so early epochs
    # stay very close to FT_LR. sanity: end-of-smoke LR should be > 0.9 × FT_LR.
    if per_epoch_lr[-1] < FT_LR * 0.5:
        print()
        print('!!' + '=' * 74)
        print(f'!! LR SANITY: end-of-epoch-{SMOKE_EPOCHS} LR = {per_epoch_lr[-1]:.2e}')
        print(f'!! Expected close to FT_LR={FT_LR:.2e} with 25-epoch schedule.')
        print('!!' + '=' * 74)

    avg_epoch_s = float(np.mean(per_epoch_s))
    extrapolated_per_seed_s = avg_epoch_s * FT_EPOCHS
    extrapolated_total_s    = extrapolated_per_seed_s * len(SEEDS)
    def _fmt_hms(s):
        s = int(round(s)); h, r = divmod(s, 3600); m, s = divmod(r, 60)
        return f'{h}h {m}m {s}s' if h else (f'{m}m {s}s' if m else f'{s}s')
    print(f'\n  Extrapolated per-seed:   {_fmt_hms(extrapolated_per_seed_s)}  '
          f'({FT_EPOCHS} epochs × {avg_epoch_s:.0f}s/epoch)')
    print(f'  Extrapolated total:      {_fmt_hms(extrapolated_total_s)}  '
          f'({len(SEEDS)} seed × {_fmt_hms(extrapolated_per_seed_s)})')

    if extrapolated_per_seed_s > 24 * 3600:
        print(f'  !! Extrapolated per-seed exceeds 24h Colab session limit.')
        print(f'  !! Resume-from-checkpoint (RESUME_FROM_CHECKPOINT=True) will let '
              f'you split across sessions.')

    if len(train_losses) >= 2 and train_losses[-1] >= train_losses[0]:
        print()
        print('!!' + '=' * 74)
        print('!! WARNING: train_loss did NOT decrease over the smoke run.')
        print(f'!!   epoch 1 loss: {train_losses[0]:.4f}')
        print(f'!!   epoch {len(train_losses)} loss: {train_losses[-1]:.4f}')
        print('!!' + '=' * 74)
    else:
        if len(train_losses) >= 2:
            print(f'\n  Train loss decreased by {train_losses[0] - train_losses[-1]:.4f} '
                  f'over smoke. lr={FT_LR} looks stable.')
        print('  Ready to flip SMOKE_ONLY=False for the full run.')

    smoke_out = Path(OUTPUT_DIR) / 'smoke_check_results.json'
    with smoke_out.open('w') as f:
        smoke_result_compact = dict(smoke_result)
        json.dump(smoke_result_compact, f, indent=2)
    print(f'\nSmoke results written: {smoke_out}')
    print(f'\nSMOKE_ONLY = {SMOKE_ONLY} — multi-seed cell below will '
          f'{"NOT train" if SMOKE_ONLY else "run the full seed"}.')


In [ ]:
# ============================================================================
# multi-seed CROMA fine-tune (3 seeds). resume-safe per seed: if a session
# dies mid-run for seed N, the next run picks up seed N from its last
# saved epoch, then continues to any un-started seeds. seeds with existing
# checkpoint_final.pt resume; seeds with no ckpt dir train fresh.
# ============================================================================
import json as _json
import numpy as np

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping the CROMA fine-tune run.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        out_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
        if out_path.exists():
            # skip training: load existing per-seed JSON into per_seed_results
            # so the aggregate step below still works.
            # if a prior run was contaminated (e.g. by the water_works
            # ASSET_TYPE_MAP bug), delete the file on Drive to force a fresh train.
            with out_path.open() as f:
                per_seed_results[seed] = _json.load(f)['finetune']
            print(f'  [SKIP] seed {seed}: existing per-seed JSON at '
                  f'{out_path.name} (delete to force rerun)')
            continue
        result = train_one_seed(seed,
                                train_set=train_global,
                                val_set=val_global,
                                test_set=test_global)
        per_seed_results[seed] = result
        out_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_seed{seed}_results.json'
        with open(out_path, 'w') as f:
            _json.dump({'finetune': result}, f, indent=2)
        print(f'\n  saved {out_path}')
        if result['resumed_epochs'] > 0:
            print(f'  (resumed from epoch {result["resumed_epochs"]}; '
                  f'session recovered {result["resumed_epochs"]} epochs of prior work)')

    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {'mean': float(arr.mean()), 'std': float(arr.std(ddof=0)),
                'per_seed': [float(v) for v in arr]}

    agg = {}
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {'class': CLASS_NAMES[i], 'idx': i,
         'mean_f1':  float(per_class_arr[:, i].mean()),
         'std_f1':   float(per_class_arr[:, i].std(ddof=0)),
         'per_seed': [float(v) for v in per_class_arr[:, i]]}
        for i in range(len(CLASS_NAMES))
    ]

    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1'] for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.mean(f1s)),
            'std_macro_f1':  float(np.std(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
               for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n':             n_seed,
            'mean_macro_f1': float(np.nanmean(f1s)),
            'std_macro_f1':  float(np.nanstd(f1s, ddof=0)),
            'per_seed':      [float(v) for v in f1s],
        }

    agg['seeds']                       = SEEDS
    agg['split_artifact']              = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note']  = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']

    agg['finetune_protocol'] = FINETUNE_PROTOCOL
    agg['peak_gpu_gb'] = {
        'mean':     float(np.mean([per_seed_results[s]['peak_gpu_gb'] for s in SEEDS])),
        'per_seed': {int(s): float(per_seed_results[s]['peak_gpu_gb']) for s in SEEDS},
    }
    agg['wall_time_s'] = {
        'mean':     float(np.mean([per_seed_results[s]['wall_time_s'] for s in SEEDS])),
        'total':    float(np.sum([per_seed_results[s]['wall_time_s'] for s in SEEDS])),
        'per_seed': {int(s): float(per_seed_results[s]['wall_time_s']) for s in SEEDS},
    }
    agg['resumed_epochs'] = {
        int(s): int(per_seed_results[s]['resumed_epochs']) for s in SEEDS
    }

    # a partial rerun must not clobber a complete three-seed aggregate
    if set(SEEDS) == set(FULL_PROTOCOL_SEEDS):
        agg_path = Path(OUTPUT_DIR) / f'{RUN_NAME_PREFIX}_aggregate.json'
        with open(agg_path, 'w') as f:
            _json.dump(agg, f, indent=2)
        print(f'\nAggregate saved: {agg_path}')
    else:
        print(f'\nNOTE: SEEDS = {SEEDS}, not the full protocol '
              f'{FULL_PROTOCOL_SEEDS}. Skipping the aggregate write '
              f'so any existing three-seed aggregate survives.')

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    cm_norm = cm_sum.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm), where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap=CONFUSION_CMAP, vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    mf1 = agg['test_macro_f1']
    ax.set_title(f'{CONFUSION_TITLE_PREFIX} — aggregate confusion (summed over '
                 f'{len(SEEDS)} seeds, row-normalized)\n'
                 f'macro F1 = {mf1["mean"]:.3f} +/- {mf1["std"]:.3f}, '
                 f'seeds = {SEEDS}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    cm_path = Path(OUTPUT_DIR) / f'confusion_matrix_{RUN_NAME_PREFIX}_aggregate.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Confusion matrix saved: {cm_path}')

    def _fmt_hms(s):
        s = int(round(s)); h, r = divmod(s, 3600); m, s = divmod(r, 60)
        return f'{h}h {m}m {s}s' if h else (f'{m}m {s}s' if m else f'{s}s')
    print('\n' + '=' * 76)
    print(f'{CONFUSION_TITLE_PREFIX} — aggregate ({len(SEEDS)} seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f}')
    print(f'Peak GPU:      {agg["peak_gpu_gb"]["mean"]:.2f} GB')
    print(f'Wall clock:    {_fmt_hms(agg["wall_time_s"]["total"])}')
    if agg['resumed_epochs'][SEEDS[0]] > 0:
        print(f'Resumed from:  epoch {agg["resumed_epochs"][SEEDS[0]]} '
              f'(session recovered)')
    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')
    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  {stats["mean_macro_f1"]:.4f}')
    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  {stats["mean_macro_f1"]:.4f}')
